# BOLD Mortality Prediction — First-Pass Pipeline

**Project:** COMFORT (personalised illness monitoring) — early pipeline development.

**What this does:** using only labs/vitals available at a *single point in time*, predict
whether an ICU patient died during their hospital admission — and explain *which* markers
drive that prediction, rather than just producing a number.

**Scope of this pass:** a simple, explainable baseline. Logistic regression + Random Forest,
SHAP for explanation, and a lightweight confounder check (does a marker's link to mortality
survive once we account for how sick the patient already was?).

**Deliberate trade-off:** the wider project's end goal is a causal framework plus a
transformer model, with an LLM layered in later. None of that is here. Step 7's confounder
check is a deliberately simple stand-in for real causal inference (no matching, IPTW, or DAGs)
— it is a first step towards separating correlation from causation, not the finished article.

## Step 0 — Data access

Real BOLD is a **credentialed PhysioNet dataset**
(https://physionet.org/content/blood-gas-oximetry/1.0/). Access needs a PhysioNet account,
completed ethics/CITI training, and a signed Data Use Agreement — still in progress.

So this pipeline currently runs on a **synthetic placeholder file** with the same column
names and realistic-looking value ranges as real BOLD, but 100% fabricated values. The point
is to get the pipeline working end-to-end, so that switching to real data is a one-line
change to `DATA_PATH` below and nothing else.

In [1]:
# ==========================================================================
#  CHANGE THIS ONE LINE when real PhysioNet access comes through:
#      DATA_PATH = "bold_dataset.csv"
#  Nothing else in this notebook needs to change.
# ==========================================================================
DATA_PATH = "synthetic_bold_dataset.csv"

# Anything that isn't the real BOLD file is synthetic, so the warning below
# switches itself off automatically once DATA_PATH points at the real data.
IS_SYNTHETIC = DATA_PATH != "bold_dataset.csv"

if IS_SYNTHETIC:
    print("=" * 78)
    print("  RUNNING ON SYNTHETIC DATA")
    print("  Every number, plot and SHAP ranking below is FABRICATED.")
    print("  These are NOT real findings - this only proves the pipeline works.")
    print("  Do not quote any result below in a report or to a supervisor.")
    print("=" * 78)
else:
    print("=" * 78)
    print("  RUNNING ON REAL BOLD DATA - results below are genuine.")
    print("=" * 78)

  RUNNING ON SYNTHETIC DATA
  Every number, plot and SHAP ranking below is FABRICATED.
  These are NOT real findings - this only proves the pipeline works.
  Do not quote any result below in a report or to a supervisor.


## Step 1 — Load and look

Load the file and get a feel for it before touching anything: how many rows and columns,
what type each column is, and how much of each column is actually filled in.

**Nothing is dropped here.** This step only reports. Deciding what to throw away comes later
(Step 4), once we've seen the picture.

In [2]:
import pandas as pd

# Show all rows when we print the missingness table - there are ~70 columns and
# pandas would otherwise hide the middle of it behind "...".
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows (patient records): {df.shape[0]:,}")
print(f"Columns:                {df.shape[1]}")

Loaded: synthetic_bold_dataset.csv
Rows (patient records): 2,000
Columns:                71


In [3]:
# What type is each column? Numbers we can model directly; "object" means text
# (e.g. race_ethnicity, source_db) and would need encoding before use.
print("Column types:")
print(df.dtypes.value_counts())
print()
print("Non-numeric (text) columns:")
text_cols = df.select_dtypes(exclude="number").columns.tolist()
print(text_cols if text_cols else "  (none)")

Column types:
float64    51
int64      18
str         2
Name: count, dtype: int64

Non-numeric (text) columns:
['source_db', 'race_ethnicity']


In [4]:
# How much of each column is actually filled in?
# In real ICU data some labs are ordered for nearly everyone (e.g. basic bloods)
# while specialist tests are only ordered when a doctor already suspects a problem -
# so missingness varies enormously between columns.
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
})
missing = missing.sort_values("pct_missing", ascending=False)

print("Missingness per column (worst first):")
print(missing.to_string())

Missingness per column (worst first):
                                 n_missing  pct_missing
others_ck_mb                          1109         55.4
others_ck_cpk                          902         45.1
hfp_bilirubin_direct                   806         40.3
others_ld_ldh                          780         39.0
coag_fibrinogen                        687         34.4
hfp_albumin                            669         33.4
hfp_alp                                653         32.6
hfp_bilirubin_total                    615         30.8
hfp_alt                                611         30.6
hfp_ast                                594         29.7
bmp_lactate                            476         23.8
coag_ptt                               413         20.6
coag_inr                               293         14.6
coag_pt                                286         14.3
cbc_rdw                                198          9.9
bmp_bun                                  0          0.0
cbc_rbc   

In [5]:
# Summarise the same thing in buckets, so it's easy to see at a glance how many
# columns are in good shape versus how many are mostly empty.
# The 60% line is where we'll cut in Step 4 - flagged here, not acted on yet.
complete    = missing[missing["pct_missing"] == 0]
usable      = missing[(missing["pct_missing"] > 0) & (missing["pct_missing"] <= 30)]
patchy      = missing[(missing["pct_missing"] > 30) & (missing["pct_missing"] <= 60)]
mostly_empty = missing[missing["pct_missing"] > 60]

print(f"Fully complete (0% missing):        {len(complete):>3} columns")
print(f"Usable (1-30% missing):             {len(usable):>3} columns")
print(f"Patchy (31-60% missing):            {len(patchy):>3} columns")
print(f"Mostly empty (>60% missing):        {len(mostly_empty):>3} columns  <- candidates to drop in Step 4")
print()

if len(mostly_empty):
    print("Columns over the 60% missing line:")
    print(mostly_empty.to_string())
else:
    print("No column is over the 60% missing line.")

Fully complete (0% missing):         56 columns
Usable (1-30% missing):               6 columns
Patchy (31-60% missing):              9 columns
Mostly empty (>60% missing):          0 columns  <- candidates to drop in Step 4

No column is over the 60% missing line.


## Step 2 — Define the label

The thing we are trying to predict (the "label" or "target") is `in_hospital_mortality`:
1 if the patient died during that hospital admission, 0 if they survived it. BOLD provides
this directly, so we don't have to derive it ourselves.

The one number that matters here is the **class balance** — what share of patients actually
died. In-hospital mortality is usually a small minority (the BOLD paper reports roughly
15-18% across its source databases), and that shapes every modelling choice that follows:
which metric we trust, and whether the model needs to be told to take the rare class
seriously.

In [6]:
# The label is already in the data as 0/1 - we don't need to construct it.
LABEL = "in_hospital_mortality"

# Sanity-check it before trusting it: it must exist, have no gaps, and contain
# only 0 and 1. A label with missing values or unexpected codes (e.g. 2, or NaN)
# would quietly corrupt everything downstream.
assert LABEL in df.columns, f"'{LABEL}' is not in the data"

n_missing_label = df[LABEL].isna().sum()
unique_values = sorted(df[LABEL].dropna().unique().tolist())

print(f"Label column:     {LABEL}")
print(f"Missing values:   {n_missing_label}")
print(f"Values present:   {unique_values}")

assert n_missing_label == 0, "Label has missing values - would need handling before modelling"
assert set(unique_values) <= {0, 1}, f"Expected only 0/1, found {unique_values}"
print("\nLabel looks clean: no gaps, values are 0/1 only.")

Label column:     in_hospital_mortality
Missing values:   0
Values present:   [0, 1]

Label looks clean: no gaps, values are 0/1 only.


In [7]:
# How many patients died vs survived?
counts = df[LABEL].value_counts().sort_index()
percents = df[LABEL].value_counts(normalize=True).sort_index() * 100

n_survived = int(counts.get(0, 0))
n_died = int(counts.get(1, 0))
pct_died = float(percents.get(1, 0.0))

print("Outcome breakdown")
print("-" * 42)
print(f"Survived (0):  {n_survived:>6,}   ({100 - pct_died:5.1f}%)")
print(f"Died     (1):  {n_died:>6,}   ({pct_died:5.1f}%)")
print("-" * 42)
print(f"Total:         {len(df):>6,}")
print()

# The imbalance ratio is the plain-English version: "for every patient who died,
# roughly N survived". Easier to say out loud than a percentage.
if n_died > 0:
    print(f"Imbalance: roughly {n_survived / n_died:.1f} survivors for every 1 death.")

Outcome breakdown
------------------------------------------
Survived (0):   1,695   ( 84.8%)
Died     (1):     305   ( 15.2%)
------------------------------------------
Total:          2,000

Imbalance: roughly 5.6 survivors for every 1 death.


In [8]:
# Why the balance above matters for what comes next:
#
# 1. ACCURACY IS A USELESS METRIC HERE. A model that predicts "everyone survives"
#    and never gets a single death right would still score the accuracy printed
#    below. So we report AUROC and AUPRC instead (Step 5), never plain accuracy.
#
# 2. THE MODEL MUST BE TOLD THE RARE CLASS MATTERS. Left alone, a model minimises
#    total error, which it can do by mostly ignoring the minority class. We use
#    class_weight="balanced" in Step 5 to make each death count proportionally more.
#
# 3. THE SPLIT MUST PRESERVE THIS RATIO. In Step 4 we stratify the train/test split
#    on this label, so the test set has the same proportion of deaths as the training
#    set - otherwise the test score reflects an easier or harder problem than the real one.

print(f"A do-nothing model predicting 'everyone survives' would score {100 - pct_died:.1f}% accuracy")
print("- which is why accuracy is not the metric we will use.")
print()

# Sense-check against what BOLD's own paper reports (roughly 15-18% mortality).
if 10 <= pct_died <= 25:
    print(f"Mortality rate of {pct_died:.1f}% is in the plausible range for ICU data.")
else:
    print(f"NOTE: {pct_died:.1f}% mortality sits outside the ~15-18% BOLD reports.")
    if IS_SYNTHETIC:
        print("Expected here - this is fabricated data, not a real cohort.")

A do-nothing model predicting 'everyone survives' would score 84.8% accuracy
- which is why accuracy is not the metric we will use.

Mortality rate of 15.2% is in the plausible range for ICU data.


## Step 3 — Pick the predictors (and avoid leakage)

Now we decide what the model is *allowed to look at*. The rule is simple: only information a
clinician would genuinely have at the bedside **at the moment of the index blood-gas reading**.

This is the step where it is easiest to fool yourself. Some columns quietly contain the answer
— they were recorded after the outcome, or are a direct consequence of it. A model given those
scores brilliantly in testing and is worthless in practice, because at the moment you'd
actually want a prediction, that information doesn't exist yet. That mistake is called
**leakage**, and it is the main thing this step exists to prevent.

Nothing is trained here. We are writing down a decision and printing it, so it can be checked
and defended rather than buried in the code.

In [9]:
# ---------------------------------------------------------------------------
# THE "ALLOWED" PILE - known at the bedside, at the time of the blood test.
# ---------------------------------------------------------------------------
# Built from column-name prefixes rather than a hand-typed list, so that any
# extra lab columns present in the real BOLD file get picked up automatically
# when we swap DATA_PATH over.

# Blood tests: full blood count, clotting, basic metabolic panel,
# liver function, and the miscellaneous enzymes.
LAB_PREFIXES = ("cbc_", "coag_", "bmp_", "hfp_", "others_")
lab_cols = [c for c in df.columns if c.startswith(LAB_PREFIXES)]

# Bedside observations. SpO2/SaO2 are oxygen saturation - part of the index
# reading itself, so they are fair game.
vital_cols = [c for c in df.columns if c.startswith("vitals_")]
vital_cols += [c for c in ["SpO2", "SaO2"] if c in df.columns]

# Blood-gas values from the index reading.
gas_cols = [c for c in ["pH", "pCO2", "pO2", "Carboxyhemoglobin", "Methemoglobin"]
            if c in df.columns]

# How sick the patient already was, scored over the PREVIOUS 24 hours.
# "past" is the key word - these look backwards from the index reading, so
# they are legitimate. Their "future" counterparts are not (see next cell).
severity_cols = [c for c in df.columns if c.startswith("sofa_past_")]

# Who the patient is, and what they came in with.
demo_cols = [c for c in ["admission_age", "sex_female", "comorbidity_score_value"]
             if c in df.columns]

for name, cols in [("Labs", lab_cols), ("Vitals", vital_cols), ("Blood gas", gas_cols),
                   ("Baseline severity", severity_cols), ("Demographics", demo_cols)]:
    print(f"{name} ({len(cols)}):")
    print(f"  {', '.join(cols)}")
    print()

Labs (32):
  cbc_wbc, cbc_hemoglobin, cbc_hematocrit, cbc_platelet, cbc_mch, cbc_mchc, cbc_mcv, cbc_rbc, cbc_rdw, coag_fibrinogen, coag_pt, coag_inr, coag_ptt, bmp_sodium, bmp_potassium, bmp_chloride, bmp_bicarbonate, bmp_bun, bmp_creatinine, bmp_glucose, bmp_aniongap, bmp_calcium, bmp_lactate, hfp_alt, hfp_alp, hfp_ast, hfp_bilirubin_total, hfp_bilirubin_direct, hfp_albumin, others_ck_cpk, others_ck_mb, others_ld_ldh

Vitals (8):
  vitals_heart_rate, vitals_resp_rate, vitals_mbp_ni, vitals_sbp_ni, vitals_dbp_ni, vitals_tempc, SpO2, SaO2

Blood gas (5):
  pH, pCO2, pO2, Carboxyhemoglobin, Methemoglobin

Baseline severity (6):
  sofa_past_coagulation_24hr, sofa_past_liver_24hr, sofa_past_cardiovascular_24hr, sofa_past_cns_24hr, sofa_past_renal_24hr, sofa_past_overall_24hr

Demographics (3):
  admission_age, sex_female, comorbidity_score_value



In [10]:
# ---------------------------------------------------------------------------
# THE "BANNED" PILE - why each group is excluded.
# ---------------------------------------------------------------------------
exclusions = {}

# 1. FUTURE SEVERITY SCORES. Measured AFTER the index reading. Using tomorrow's
#    information to predict tomorrow is not prediction, it is reading the answer.
exclusions["sofa_future_* (measured after the index reading)"] = [
    c for c in df.columns if c.startswith("sofa_future_")
]

# 2. LENGTH OF STAY. Only known once the admission is over - so it isn't available
#    at prediction time at all. It is also a direct giveaway: patients who die often
#    have a short stay, so the model would learn to read the outcome backwards.
exclusions["length of stay (only known once the admission ends)"] = [
    c for c in ["los_hospital", "los_ICU"] if c in df.columns
]

# 3. DISCHARGE / TIMESTAMP FIELDS. Same problem - they sit at or after the outcome.
#    None are present in this synthetic file, but the rule is written by pattern so
#    it will catch them automatically in the real BOLD data.
exclusions["timestamps and discharge fields (at or after the outcome)"] = [
    c for c in df.columns
    if "_timestamp" in c or "datetime_" in c or "discharge" in c.lower()
]

# 4. IDENTIFIERS AND SOURCE. Admission numbers carry no clinical meaning, but a model
#    can still latch onto accidental patterns in them - that is noise, not signal.
#    source_db just records which hospital database a row came from.
exclusions["identifiers and source database (no clinical meaning)"] = [
    c for c in ["unique_subject_id", "unique_hospital_admission_id",
                "unique_icustay_id", "source_db"] if c in df.columns
]

# 5. THE LABEL ITSELF, plus race_ethnicity - which we hold back from the model but
#    keep in the data, to check performance across groups in Step 9.
exclusions["the outcome, and race_ethnicity (held back for the Step 9 fairness check)"] = [
    c for c in [LABEL, "race_ethnicity"] if c in df.columns
]

# 6. BODY MEASUREMENTS. NOTE - these are NOT leaky: height, weight and BMI are all
#    recorded at admission, so they'd be available at prediction time. They are left
#    out only because the agreed predictor list for this first pass didn't include
#    them. If we want them, move this line up into demo_cols - no other change needed.
exclusions["body measurements (not leaky - just outside the agreed first-pass list)"] = [
    c for c in ["weight_admission", "height_admission", "BMI_admission"] if c in df.columns
]

for reason, cols in exclusions.items():
    if cols:
        print(f"EXCLUDED - {reason}")
        print(f"  {', '.join(cols)}")
        print()

EXCLUDED - sofa_future_* (measured after the index reading)
  sofa_future_coagulation_24hr, sofa_future_liver_24hr, sofa_future_cardiovascular_24hr, sofa_future_cns_24hr, sofa_future_renal_24hr, sofa_future_overall_24hr

EXCLUDED - length of stay (only known once the admission ends)
  los_hospital, los_ICU

EXCLUDED - identifiers and source database (no clinical meaning)
  unique_subject_id, unique_hospital_admission_id, unique_icustay_id, source_db

EXCLUDED - the outcome, and race_ethnicity (held back for the Step 9 fairness check)
  in_hospital_mortality, race_ethnicity

EXCLUDED - body measurements (not leaky - just outside the agreed first-pass list)
  weight_admission, height_admission, BMI_admission



In [11]:
# ---------------------------------------------------------------------------
# Assemble the final feature list, and prove nothing was missed.
# ---------------------------------------------------------------------------
FEATURES = lab_cols + vital_cols + gas_cols + severity_cols + demo_cols

# Every column must land in exactly one pile. If a column is in neither, it was
# silently forgotten - which is exactly how a leaky column sneaks into a model.
all_excluded = {c for cols in exclusions.values() for c in cols}
unaccounted = [c for c in df.columns if c not in FEATURES and c not in all_excluded]

print(f"Total columns in data:  {len(df.columns)}")
print(f"Kept as predictors:     {len(FEATURES)}")
print(f"Excluded:               {len(all_excluded)}")
print(f"Unaccounted for:        {len(unaccounted)}")
print()

if unaccounted:
    print("WARNING - these columns were neither kept nor explicitly excluded:")
    print(f"  {', '.join(unaccounted)}")
    print("Decide on each one before continuing.")
else:
    print("Every column is accounted for - nothing was silently dropped or included.")

# A blunt backstop: if any known-leaky name pattern survived into FEATURES,
# stop immediately rather than train a model that looks good and is worthless.
leak_patterns = ("sofa_future_", "los_", "_timestamp", "datetime_", "discharge")
leaked = [c for c in FEATURES if any(p in c for p in leak_patterns)]
assert not leaked, f"LEAKAGE - these must not be predictors: {leaked}"
assert LABEL not in FEATURES, "LEAKAGE - the outcome itself is in the feature list"
print("Leakage check passed.")

Total columns in data:  71
Kept as predictors:     54
Excluded:               17
Unaccounted for:        0

Every column is accounted for - nothing was silently dropped or included.
Leakage check passed.
